# A large scale analysis of conversation data across myriad spoken corpora

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [ ]:
bse = .002257

In [ ]:
DATA_LOC = 'data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join(DATA_LOC, 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join(VIS_PATH, 'final-document.tsv')

In [ ]:
def grab_all_files(PATH, file_type='.csv'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

In [ ]:
# to rename the corpus correctly . . . 
def rename(x):
    result = x
    if x.startswith('se'):
        result = 'CORAAL'
        
    if x.startswith('call'):
        result = x.split('-')[0]
    
    if x.startswith('MICASE'):
        result = 'MICASE'
    
    if x.startswith('instruction'):
        result = x.split('-')[-1]
    
    if x.startswith('CABNC'):
        result = 'CABNC'
    
    return result

In [ ]:
df_ = pd.read_table(FINAL_DOC, sep='\t')

In [ ]:
df_['corpus'] = [rename(co) for co in tqdm(df_['corpus'])]

In [ ]:
df_.head()

## Vis & Other Tests

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
names = ['other', 'self']
main_line_colors = ['#1f77b4','#ff7f0e']
ribbon_colors = ['rgba(31, 119, 180, 0.2)', 'rgba(255, 127, 14, 0.2)']

#### Initial plots

In [ ]:
# sns.lineplot(
#     data=df_.loc[
#         ~df_['self']
#         & (df_['turn_delta'] > 0)
#         & (df_['turn_delta'] < 40)
#     ], 
#     y='resid', 
#     x='turn_delta', 
# )
# plt.show()

In [ ]:
# sns.lineplot(
#     data=df_.loc[
#         df_['self'] 
#         & (df_['turn_delta'] > 0)
#         & (df_['turn_delta'] < 40)
#     ], 
#     y='resid', 
#     x='turn_delta', 
#     color='orange'
# )
# plt.show()

In [ ]:
ax = sns.lineplot(
    data=df_.loc[
        (df_['turn_delta'] > 0)
        & (df_['turn_delta'] < 40)
    ], 
    y='resid', 
    x='turn_delta', 
    hue='self',
    legend=False
)
plt.show()

### Creating an interactive plot using px to start, and then converting to GO :)

In [ ]:
import plotly.tools as tls
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
fig = tls.mpl_to_plotly(ax.figure)

In [ ]:
new_fig = go.Figure()

In [ ]:
for i,trace in enumerate(fig.data):
    trace['name'] = names[i]
    # print(trace)
    trace['showlegend'] = True
    trace['line']['color'] = main_line_colors[i]
    trace['legendgroup'] = names[i]
    # trace['legendgrouptitle'] = names[i]
    new_fig.add_trace(trace)
    
    # upper bounds for ribbon
    new_fig.add_trace(
        go.Scatter(
            x=trace['x'],
            y=(np.array(trace['y']) + (bse*2)).tolist(),
            mode='lines',
            line=dict(color=main_line_colors[i],width=0),
            name=names[i]+'_upper_bound',
            showlegend=False,
            legendgroup=names[i]
        )
    )
    
    
    # lower bounds for ribbon
    new_fig.add_trace(
        go.Scatter(
            x=trace['x'],
            y=(np.array(trace['y']) - (bse*2)).tolist(),
            mode='lines',
            line=dict(color=main_line_colors[i],width=0),
            name=names[i]+'_lower_bound',
            fill='tonexty',
            fillcolor=ribbon_colors[i],
            showlegend=False,
            legendgroup=names[i]
        )
    )
    
# new_fig.update_layout(showlegend=True)
new_fig.update_layout(template='plotly')
new_fig.show()  

In [ ]:
new_fig.write_html(os.path.join(VIS_PATH, 'entropy-delta.html'))

### Individual conversations in a subplot

In [ ]:
from plotly.subplots import make_subplots

In [ ]:
def create_subplot(
        dataframe, 
        new_fig, 
        row, 
        col, 
        names=names, 
        main_line_colors=main_line_colors, 
        ribbon_colors=ribbon_colors
):
    
    ax = sns.lineplot(
        data=dataframe, 
        y='resid', 
        x='turn_delta', 
        hue='self',
        legend=False
    )
    
    fig = tls.mpl_to_plotly(ax.figure)
    
    for i,trace in enumerate(fig.data):
        trace['name'] = names[i]
        # print(trace)
        trace['showlegend'] = False
        if (row==1) and (col==1):
            trace['showlegend'] = True
        trace['line']['color'] = main_line_colors[i]
        trace['legendgroup'] = names[i]
        # trace['legendgrouptitle'] = names[i]
        new_fig.add_trace(trace, row=row, col=col)
        
        # upper bounds for ribbon
        new_fig.add_trace(
            go.Scatter(
                x=trace['x'],
                y=(np.array(trace['y']) + bse*2).tolist(),
                mode='lines',
                line=dict(color=main_line_colors[i],width=0),
                name=names[i]+'_upper_bound',
                showlegend=False,
                legendgroup=names[i]
            ), row=row, col=col
        )
        
        # lower bounds for ribbon
        new_fig.add_trace(
            go.Scatter(
                x=trace['x'],
                y=(np.array(trace['y']) - bse*2).tolist(),
                mode='lines',
                line=dict(color=main_line_colors[i],width=0),
                name=names[i]+'_lower_bound',
                fill='tonexty',
                fillcolor=ribbon_colors[i],
                showlegend=False,
                legendgroup=names[i]
            ), row=row, col=col
        )

#### Actually creating the figure

In [ ]:
conversation_id_col = 'conversation_id'

In [ ]:
k_conversations = 12
k_columns = 3
k_rows = int(np.ceil(k_conversations/k_columns))

In [ ]:
assignments = sum([[(row+1,col+1) for col in range(k_columns)] for row in range(k_rows)], [])

### Specific Corpus Visualization

In [ ]:
df_['corpus'].value_counts()

In [ ]:
SPECIFIC_CORPUS = 'GCSAusE'
sel = df_['corpus'].isin([SPECIFIC_CORPUS])
sel.sum()

In [ ]:
# conversations
subdata = np.random.choice(df_[conversation_id_col].loc[sel].unique(), size=(k_conversations,), replace=False).tolist()

In [ ]:
tracking_doc = os.path.join(VIS_PATH,'tracking.txt')

if (not os.path.exists(tracking_doc)):
    f = open(tracking_doc, 'w')
    f.write('{} | {} | {}\n\n'.format(dt.now(), SPECIFIC_CORPUS, str(subdata)))
    f.close()

else:
    f = open(tracking_doc, 'a')
    f.write('{} | {} | {}\n\n'.format(dt.now(), SPECIFIC_CORPUS, str(subdata)))
    f.close()

In [ ]:
subplot_fig = make_subplots(
    rows=k_rows,cols=k_columns, x_title=SPECIFIC_CORPUS,
    # subplot_titles=subdata
    # row_titles=['Zoom', 'Telephone', 'Instruction', 'BNC/Naturalistic'],
    # row_titles=['Instruction','Instruction', 'BNC/Naturalistic','BNC/Naturalistic'],
)

In [ ]:
for (i, conversation) in tqdm(enumerate(subdata)):
    df__ = df_.loc[
        df_[conversation_id_col].isin([conversation])
        & (df_['turn_delta'] > 0)
        & (df_['turn_delta'] < 40)
    ]
    
    create_subplot(
        dataframe=df__,
        new_fig=subplot_fig,
        row=assignments[i][0],
        col=assignments[i][1]
    )   

In [ ]:
subplot_fig.update_layout(template='plotly')
subplot_fig.show()

In [ ]:
subplot_fig.write_html(os.path.join(VIS_PATH, 'ind_corpora', 'entropy-delta-individual-conversations-{}.html'.format(SPECIFIC_CORPUS)))